# Feature engineering: into the Feast store

The business features are already clean census aggregates — engineering here
means **publishing them as store objects**: entity keys, a parquet source, a
feature view online callers can hit, and a labels view for training.

## Bring up the local AWS stack

One moto container serves S3/STS **and DynamoDB** — the same endpoint
Feast's `dynamodb` online store writes to.

In [ ]:
%%bash
cat docker-compose.yml

In [ ]:
%%bash
make up-moto

In [ ]:
%%bash
make ps

## Build the parquet tables

In [ ]:
%%bash
python -m california_house_pricing.prepare_data

## Publish them into the feature repo

In [ ]:
%%bash
cp data/processed/block_group_features.parquet data/processed/house_values.parquet feature_repo/data/ 2>/dev/null || (mkdir -p feature_repo/data && cp data/processed/*.parquet feature_repo/data/)

In [ ]:
%%bash
cd feature_repo && feast apply

`feast apply` registered the entity, both views, and — because
`online_store.type: dynamodb` — created the online table on moto.

In [ ]:
%%bash
aws dynamodb list-tables --endpoint-url http://california-house-pricing-moto:5000

## Materialize online features

In [ ]:
import subprocess

subprocess.run(
    ["feast", "materialize-incremental", "9999-12-31T00:00:00"],
    cwd="feature_repo",
    check=False,
)
# full-range materialize for the single snapshot

In [ ]:
%%bash
cd feature_repo && feast materialize 1990-01-01T00:00:00 2030-01-01T00:00:00

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")
resp = store.get_online_features(
    features=[
        "block_group_profile:median_income",
        "block_group_profile:house_age",
    ],
    entity_rows=[{"block_group_id": "bg-00000"}],
).to_dict()
resp

The online value came back through DynamoDB — the same row the parquet
table holds. Store consistency verified before a single training run.

## Point-in-time training frame

In [ ]:
import pandas as pd

entities = pd.read_parquet("data/processed/house_values.parquet")
training = store.get_historical_features(
    entity_df=entities,
    features=[
        "block_group_profile:median_income",
        "block_group_profile:house_age",
        "block_group_profile:avg_rooms",
        "block_group_profile:avg_bedrooms",
        "block_group_profile:population",
        "block_group_profile:avg_occupancy",
        "block_group_profile:latitude",
        "block_group_profile:longitude",
        "house_value_labels:median_house_value",
    ],
).to_df()
training.head()